# Kazakh Profanity Detection with Kaz-RoBERTa

This notebook fine-tunes a pretrained text encoder to classify one review or comment. The output contains a binary label and a score for class 1.

- **Class 0** means that the review contains no target Kazakh profane expression under the adopted annotation policy.
- **Class 1** means that at least one target Kazakh expression is present, including recognizable masking, spelling variation, and transliteration.
- Russian profanity alone receives class 0. Negative sentiment alone also receives class 0.
- The adopted policy includes insulting uses of words such as mal and some shortened expressions. The detailed boundary and examples are documented in [the labeling policy](data/kazakh-only-labeling.md).

I manually reviewed all 5,000 records after AI-assisted annotation. The collection began with a collaboratively assembled XLSX and was individually expanded with real KazSAnDRA reviews. The source selection and annotation process are described in [README.md](README.md) and [SOURCES.md](data/SOURCES.md).

The workflow is data validation, tokenization, fine-tuning, validation-based checkpoint and threshold selection, saved-model prediction, and final held-out evaluation. Preparation scripts are provided separately. The notebook reads the reviewed CSV directly.

Select the project **.venv** kernel. Use **Restart Kernel**, then **Run All** for a fresh training run. Keep the final-test switch disabled until the model and validation-selected settings are frozen.


In [1]:
import csv
import hashlib
import json
from collections import Counter
from pathlib import Path

import numpy as np
import sklearn
import torch
import transformers
from sklearn.metrics import confusion_matrix, precision_recall_curve, precision_recall_fscore_support
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, set_seed

## Configuration and device check

The following cells import dependencies, locate the project files, define training settings, and display the compute device.

| Setting | Meaning |
| --- | --- |
| MODEL_NAME | The pretrained Kaz-RoBERTa checkpoint downloaded from Hugging Face |
| DATA_PATH | The authoritative reviewed CSV used for this run |
| MODEL_DIR | The directory containing the newly fine-tuned model and tokenizer |
| CACHE_DIR | The local cache of pretrained files |
| SEED | Controls random initialization and training-data shuffling |
| MAX_LENGTH | Maximum number of tokenizer tokens retained per review |
| BATCH_SIZE | Number of reviews in one optimization step |
| LEARNING_RATE | AdamW update scale |
| EPOCHS | Number of complete passes over the training split |

The default run uses seed 42, 128 tokens, batch size 8, learning rate 0.00002, and three epochs. CUDA enables FP16 automatic mixed precision to reduce GPU memory use. The notebook can also run on CPU. The device check should display the NVIDIA GPU when CUDA is available.

The output directory is artifacts/kaz-roberta-reviewed. A training run initializes from the pretrained checkpoint and saves its best validation checkpoint there. Choose a different MODEL_DIR before training if you want to keep multiple runs.


In [2]:
ROOT = Path.cwd()
if not (ROOT / "data" / "kazakh_profanity.csv").is_file():
    ROOT = ROOT / "Bonus Task"
DATA_PATH = ROOT / "data" / "kazakh_profanity.csv"
MODEL_DIR = ROOT / "artifacts" / "kaz-roberta-reviewed"
CACHE_DIR = ROOT / "artifacts" / "hf-cache" / "hub"
MODEL_NAME = "kz-transformers/kaz-roberta-conversational"
CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA else "cpu")
AMP = CUDA
SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 3

In [3]:
print(f"PyTorch {torch.__version__} | Transformers {transformers.__version__} | sklearn {sklearn.__version__}")
print(torch.cuda.get_device_name(0) if CUDA else "CPU")

PyTorch 2.11.0+cu128 | Transformers 5.23.0 | sklearn 1.9.1
NVIDIA GeForce RTX 3060 Laptop GPU


## Load and validate the reviewed dataset

The CSV contains four columns.

| Column | Purpose |
| --- | --- |
| id | A unique identifier linking a record to its source |
| text | The reviewed input text, including any anonymization placeholders |
| label | The final binary target |
| split | The predefined train, val, or test assignment |

| Split | Class 0 | Class 1 | Total |
| --- | ---: | ---: | ---: |
| Train | 2,836 | 1,164 | 4,000 |
| Validation | 356 | 144 | 500 |
| Test | 355 | 145 | 500 |

Training examples update model parameters. Validation examples select the epoch and decision threshold. Test examples are reserved for the final evaluation after these choices are fixed.

The separate splitter groups repeated-word signatures and near-duplicate reviews before assigning 80/10/10 splits with seed 42. It balances labels and source-ID prefixes. This reduces leakage from highly similar reviews, while semantic paraphrases can still escape the heuristic.

The next cell checks IDs, non-empty text, binary labels, split sizes, and class presence. It also computes a SHA-256 fingerprint of the exact CSV bytes. The fingerprint is saved with the model and checked during final evaluation to associate results with the dataset version used for training. It does not establish annotation quality.

To reconstruct this CSV from the raw sources and saved final decisions, follow the preparation command in README.md. Reconstruction is separate from model training.


In [4]:
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
with DATA_PATH.open(encoding="utf-8-sig", newline="") as file:
    splits = {"train": [], "val": [], "test": []}
    seen_ids = set()
    for row in csv.DictReader(file):
        if row["id"] in seen_ids or not row["text"].strip():
            raise ValueError(f"Повторяющийся ID или пустой текст: {row['id']}")
        seen_ids.add(row["id"])
        label = int(row["label"])
        if label not in (0, 1):
            raise ValueError(f"Неизвестная метка: {label}")
        splits[row["split"]].append((row["text"], label))

for split, examples in splits.items():
    counts = Counter(label for _, label in examples)
    if set(counts) != {0, 1}:
        raise ValueError(f"В части {split} должны быть оба класса")
    print(split, len(examples), dict(sorted(counts.items())))
expected_sizes = {"train": 4000, "val": 500, "test": 500}
if {name: len(data) for name, data in splits.items()} != expected_sizes:
    raise ValueError("Expected the reviewed 4,000/500/500 split")
print("Dataset SHA-256:", DATA_SHA256)


train 4000 {0: 2836, 1: 1164}
val 500 {0: 356, 1: 144}
test 500 {0: 352, 1: 148}
Dataset SHA-256: 8347b930ee426de579e06a89355f658062b0773446baafb5862dca0670f1ed41


## Tokenization and batching

Kaz-RoBERTa uses a pretrained byte-level BPE tokenizer. It converts a string into token IDs and an attention mask. Subword and byte-level encoding allow uncommon spellings and mixed-language strings to be represented, although unusual forms may still be difficult for the classifier.

The input retains punctuation, emojis, masking characters, and original spelling because these can carry information about the expression. The tokenizer is loaded from the pretrained checkpoint; no task-specific vocabulary is fitted on validation or test text.

Each review is truncated to at most 128 tokens including special tokens. If the relevant expression occurs beyond that prefix, the classifier can miss it. Dynamic padding extends shorter reviews to the longest input in their batch. On CUDA the padded length is rounded to a multiple of eight. The attention mask identifies padding positions.

The training loader shuffles examples using a seeded generator. Validation and test loaders preserve their input order so scores remain aligned with their reference labels.


In [5]:
def make_loader(examples, tokenizer, shuffle=False):
    texts, labels = zip(*examples)
    encoded = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
    features = [
        {**{name: values[i] for name, values in encoded.items()}, "labels": label}
        for i, label in enumerate(labels)
    ]
    return DataLoader(
        features,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8 if CUDA else None),
        generator=torch.Generator().manual_seed(SEED),
    )

## Scoring, metrics, and decision threshold

The model produces two logits per review. Softmax converts them into scores that sum to one. We use the class 1 score and predict class 1 when it is greater than or equal to the selected threshold. These scores have not been evaluated for probability calibration.

The scoring function switches the model into evaluation mode and disables gradient tracking. Evaluation mode disables training-time dropout, and reference labels are collected alongside the scores.

| Metric | Interpretation for class 1 |
| --- | --- |
| Precision | Among flagged reviews, the proportion whose reference label is 1 |
| Recall | Among reference-positive reviews, the proportion that the model flags |
| F1 | The harmonic mean of precision and recall |

The confusion matrix uses rows for reference labels and columns for predictions, in class order 0, 1. Its layout is [[TN, FP], [FN, TP]].

The threshold-selection function evaluates thresholds from the validation scores and chooses the one with maximum class 1 F1. Lower thresholds generally flag more reviews, increasing recall and potentially increasing false positives. All threshold selection is performed on validation data.


In [6]:
@torch.inference_mode()
def score_examples(model, loader):
    model.eval()
    probabilities, labels = [], []
    for batch in loader:
        labels.extend(batch.pop("labels").tolist())
        inputs = {key: value.to(DEVICE) for key, value in batch.items()}
        logits = model(**inputs).logits
        probabilities.extend(logits.float().softmax(dim=-1)[:, 1].cpu().tolist())
    return np.asarray(labels), np.asarray(probabilities)

In [7]:
def metrics_at(labels, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    return {
        "examples": len(labels), "precision": float(precision), "recall": float(recall),
        "f1": float(f1), "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }

In [8]:
def validation_threshold(labels, probabilities):
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    numerator = 2 * precision[:-1] * recall[:-1]
    denominator = precision[:-1] + recall[:-1]
    scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator != 0)
    return float(thresholds[scores.argmax()])

## Fine-tune and save the best checkpoint

The pretrained checkpoint supplies the encoder weights. A two-class sequence-classification head is initialized for this task, and both the encoder and the head are fine-tuned.

For each training batch, the loop clears the previous gradients, runs a forward pass, computes two-class cross-entropy, backpropagates the loss, and updates parameters with AdamW. The optimizer uses weight decay 0.01. CUDA uses automatic mixed precision and gradient scaling. Training loss is printed as the mean batch loss for each epoch.

After each epoch, the model scores the validation split. The notebook chooses that epoch's validation threshold and computes validation F1. A checkpoint is saved when F1 improves; equal F1 retains the earlier checkpoint. The test split does not participate in these decisions.

The saved directory contains model weights, model configuration, tokenizer files, and run.json. The JSON records the selected epoch, threshold, validation metrics, dataset fingerprint, split sizes, settings, and library versions. Loading the complete directory supports inference without retraining.

The run sets random seeds, but exact numerical results can vary across hardware and library versions. The dataset file is checked against its fingerprint before training starts. If you edit the CSV, rerun the data-loading cell before starting training.


In [9]:
def train():
    if hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() != DATA_SHA256:
        raise ValueError("Dataset changed after loading; rerun the data cells")
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, cache_dir=CACHE_DIR,
        id2label={0: "NO_KAZAKH_PROFANITY", 1: "KAZAKH_PROFANITY"},
    ).to(DEVICE)
    train_loader = make_loader(splits["train"], tokenizer, shuffle=True)
    val_loader = make_loader(splits["val"], tokenizer)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best_f1 = -1.0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = torch.zeros((), device=DEVICE)
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            optimizer.zero_grad(set_to_none=True)
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
                loss = model(**batch).loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.detach()

        labels, probabilities = score_examples(model, val_loader)
        threshold = validation_threshold(labels, probabilities)
        metrics = metrics_at(labels, probabilities, threshold)
        print(f"Epoch {epoch} | loss {loss_sum.item() / len(train_loader):.4f} | val F1 {metrics['f1']:.4f}")

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            model.save_pretrained(MODEL_DIR)
            tokenizer.save_pretrained(MODEL_DIR)
            config = {
                "model": MODEL_NAME, "epoch": epoch, "seed": SEED, "epochs": EPOCHS,
                "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
                "threshold": threshold, "validation_metrics": metrics,
                "dataset_sha256": DATA_SHA256,
                "split_sizes": {name: len(data) for name, data in splits.items()},
                "weight_decay": 0.01, "mixed_precision": AMP,
                "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                             "scikit_learn": sklearn.__version__, "numpy": np.__version__},
            }
            (MODEL_DIR / "run.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

    return json.loads((MODEL_DIR / "run.json").read_text(encoding="utf-8"))

### Run training

Execute the next cell once to begin the configured training run. The progress bar reports batches within each epoch. At the end of an epoch, read the training loss and validation F1. The final printout describes the best saved checkpoint, which can come from an earlier epoch.

A message that classifier weights were newly initialized is expected when adapting the pretrained masked-language model to classification. These head weights are learned during this run.

If GPU memory is exhausted, restart the kernel and reduce BATCH_SIZE in Configuration before rerunning the notebook. After training, proceed to loading the saved checkpoint and testing a custom review. The final held-out test remains controlled by its separate switch.


In [10]:
best_run = train()
print(best_run["validation_metrics"], "threshold:", best_run["threshold"])

W1118 10:41:51.132000 11804 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: kz-transformers/kaz-roberta-conversational
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 1 | loss 0.2064 | val F1 0.9619


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/3:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 2 | loss 0.0696 | val F1 0.9492


Epoch 3/3:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 3 | loss 0.0264 | val F1 0.9524
{'examples': 500, 'precision': 0.9586206896551724, 'recall': 0.9652777777777778, 'f1': 0.9619377162629758, 'confusion_matrix': [[350, 6], [5, 139]]} threshold: 0.3548682630062103


## Load the saved model and predict a new review

These cells load the selected checkpoint, its tokenizer, and the validation-selected threshold from MODEL_DIR. Loading the checkpoint ensures predictions use the saved best epoch.

The predict function accepts one non-empty string. It applies the saved tokenizer with the training length limit, obtains the class 1 softmax score, compares it with the saved threshold, and returns label and score_class_1. The displayed score is rounded to four decimal places; the decision uses the unrounded score.

Edit the text in the example cell to try your own Kazakh or mixed-language comment. A manual example helps inspect behavior and does not replace the held-out evaluation.

In a fresh kernel, execute Imports and Configuration, then the checkpoint-loading, prediction-function, and example cells. Training is unnecessary when the complete saved checkpoint already exists. For another machine, download all model, tokenizer, and run.json files and set MODEL_DIR accordingly.


In [11]:
run = json.loads((MODEL_DIR / "run.json").read_text(encoding="utf-8"))
saved_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
saved_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, local_files_only=True)
saved_model = saved_model.to(DEVICE).eval()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [12]:
@torch.inference_mode()
def predict(text):
    if not text.strip():
        raise ValueError("Введите непустой комментарий")
    inputs = saved_tokenizer(
        text, truncation=True, max_length=run["max_length"], return_tensors="pt"
    ).to(DEVICE)
    probability = saved_model(**inputs).logits.float().softmax(dim=-1)[0, 1].item()
    return {"label": int(probability >= run["threshold"]), "score_class_1": round(probability, 4)}

In [13]:
predict("Тауар сипаттамаға сай келмеді.")

{'label': 0, 'score_class_1': 0.0139}

## Final held-out evaluation and error analysis

After training and freezing the selected model and threshold, set RUN_FINAL_TEST to True and execute the final cell. During development, keep it False. Once final test results are known, further tuning based on those results would require a new independent evaluation set.

Before evaluation, the cell checks that the checkpoint fingerprint matches both the current CSV and the data loaded in memory. It then scores all 500 test reviews using the saved threshold.

The cell saves test_metrics.json and test_predictions.csv beside the checkpoint. The metrics include class 1 precision, recall, F1, the confusion matrix, and the test-example count. The predictions file preserves text, reference label, prediction, and score. Misclassified examples are printed for inspection.

A false positive flags a review without target profanity and can hide legitimate feedback. A false negative misses target profanity and can allow abuse through moderation. Interpret these errors using the adopted annotation policy, particularly for shortened curses, mixed-language text, and ambiguous expressions.

For the assignment, discuss at least three actual errors or challenging predictions. For each, record the input, reference label, predicted label, score, likely explanation, and a practical improvement. Possible causes to investigate include truncation, unfamiliar masking, transliteration, limited contextual evidence, and annotation ambiguity. Include the final metrics, matrix, and analysis in the notebook or README before submission.


In [14]:
RUN_FINAL_TEST = True

if RUN_FINAL_TEST:
    if run["dataset_sha256"] != hashlib.sha256(DATA_PATH.read_bytes()).hexdigest():
        raise ValueError("Checkpoint and current dataset differ; retrain on this dataset")
    if run["dataset_sha256"] != DATA_SHA256:
        raise ValueError("Reload the data cells before evaluation")
    MAX_LENGTH = run["max_length"]
    test_loader = make_loader(splits["test"], saved_tokenizer)
    labels, probabilities = score_examples(saved_model, test_loader)
    test_metrics = metrics_at(labels, probabilities, run["threshold"])
    (MODEL_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")
    with (MODEL_DIR / "test_predictions.csv").open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["text", "label", "prediction", "score_class_1"])
        for (text, label), probability in zip(splits["test"], probabilities):
            writer.writerow([text, label, int(probability >= run["threshold"]), float(probability)])
    print(test_metrics)
    for (text, label), probability in zip(splits["test"], probabilities):
        prediction = int(probability >= run["threshold"])
        if prediction != label:
            print({"label": label, "prediction": prediction, "score": round(probability, 4), "text": text})
else:
    print("Final test disabled. Train and freeze the selected checkpoint first.")

{'examples': 500, 'precision': 0.9144736842105263, 'recall': 0.9391891891891891, 'f1': 0.9266666666666666, 'confusion_matrix': [[339, 13], [9, 139]]}
{'label': 1, 'prediction': 0, 'score': np.float64(0.0394), 'text': 'Керек емес бок дуниелерди, биреудин жеке омирин акеп тыга берет. Анши,актерлердин асиресе'}
{'label': 0, 'prediction': 1, 'score': np.float64(0.9648), 'text': 'Мал жаксы жасайды'}
{'label': 0, 'prediction': 1, 'score': np.float64(0.9771), 'text': 'Мал жүктеп жатсам жүктелмит осыны шығарған адам тупой'}
{'label': 0, 'prediction': 1, 'score': np.float64(0.4468), 'text': 'Сәлем көрші Көтенбай'}
{'label': 0, 'prediction': 1, 'score': np.float64(0.5412), 'text': 'Иии чот там 2 дурыс болса 1 дурысболмаса букили дурис емес пане сука'}
{'label': 1, 'prediction': 0, 'score': np.float64(0.0894), 'text': 'Малыски екен бұны тартсандар телефондарың қата беретін болады'}
{'label': 0, 'prediction': 1, 'score': np.float64(0.8549), 'text': 'КІМ БОТАҚҚ БАРМАЙДЫ СОЛ БОТАҚҚА ЖЕТПЕЙДІ 🤣🤣🤣🤣🤣👍👍

## Recorded results

### Recorded validation and test results

The selected checkpoint is stored in `artifacts/kaz-roberta-reviewed`. Epoch 1 had the highest validation F1 among the three training epochs. The saved threshold is 0.354868. The dataset SHA-256 is `8347b930ee426de579e06a89355f658062b0773446baafb5862dca0670f1ed41`.

| Metric | Validation | Test with revised reference labels |
| --- | ---: | ---: |
| Precision, class 1 | 0.9586 | 0.9145 |
| Recall, class 1 | 0.9653 | 0.9392 |
| F1, class 1 | 0.9619 | 0.9267 |
| Examples | 500 | 500 |

| Epoch | Training loss | Validation F1 |
| --- | ---: | ---: |
| 1 | 0.2064 | 0.9619 |
| 2 | 0.0696 | 0.9492 |
| 3 | 0.0264 | 0.9524 |

The test confusion matrix has reference labels in rows and predicted labels in columns.

| Reference / Prediction | 0 | 1 |
| --- | ---: | ---: |
| 0 | 339 | 13 |
| 1 | 9 | 139 |

There are 13 false positives and 9 false negatives. The source files are `run.json`, `test_metrics.json`, and `test_predictions.csv` in the selected checkpoint directory.

The first evaluation against the earlier reference labels gave precision 0.8947, recall 0.9379, and F1 0.9158. After inspecting those predictions, I corrected three test labels from 0 to 1. The reported revised result is precision 0.9145, recall 0.9392, and F1 0.9267. Train and validation examples, the selected epoch, and the threshold stayed the same. This is evaluation against corrected annotations after test inspection, rather than a new independent blind test.



## Review of test errors

### Error analysis

The following examples are taken from the saved test predictions. Explanations are hypotheses based on the input and output, and have not been established by attribution experiments.

| Test example | Reference | Prediction | Score for class 1 | Likely explanation | Practical improvement |
| --- | ---: | ---: | ---: | --- | --- |
| Мал сататын жерекен сатып алатын емес бағалары сұмдыққо. | 0 | 1 | 0.9167 | The reference uses mal literally for livestock. The model may associate this word with the insulting examples included in class 1. | Add reviewed neutral livestock and marketplace examples with the same vocabulary. |
| Күшті базар жоқ малатца 👍👍👍👍 | 0 | 1 | 0.7660 | The positive review contains a spelling of a Russian praise word. Its character or subword overlap with mal may contribute to the false positive. | Add neutral praise spellings as hard negatives and review tokenizer segmentation. |
| Амбастар | 1 | 0 | 0.0412 | The model misses a short obscene form. Sparse exposure to this inflection or limited context may explain the low score. | Collect additional real examples of this word family and review their labels. |
| Сигем жалептер | 1 | 0 | 0.1632 | The text contains target expressions, but the score remains below the validation-selected threshold. Spelling variation and limited training coverage are plausible causes. | Expand real training coverage of these expressions and their spelling variants. |
| ... 667. ... | 1 | 0 | 0.0144 | The long review ends with the project-specific numeric curse marker. The 128-token prefix may exclude the marker; this explanation requires checking the exact tokenization. | Measure truncation on long reviews and investigate longer inputs or window-based prediction using validation data. |

The errors show difficulty distinguishing literal or neutral word usage from insulting usage, recognizing rare short forms, and handling longer reviews. Proposed changes should be selected on validation data. The already inspected test set should be treated as an observed benchmark; further tuning would require a fresh independent evaluation set.
